<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Warum LangGraph?
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Hier wird sichtbar, warum der Meeting- & Briefing-Agent mehr braucht als eine LLM-Schleife: explizites **Planen** (State und Routing statt "hoffentlich merkt sich das LLM den Kontext") und kontrolliertes **Handeln** (Graph-Schritte statt impliziter LLM-Wahl). Das ist der Kernunterschied zu einem reinen GenAI-Chatbot mit Tools.

**Praxisanker:** Mara Vogt profitiert von LangGraph, wenn aus einer einzelnen Briefing-Frage ein nachvollziehbarer Ablauf mit State, Routing und Prüfpunkt wird.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "m07-warum-langgraph"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["m07", "langgraph"],
    "metadata": {"notebook": "m07", "version": "1.0"}
}


# 1 | Limitierungen von create_agent()
---


`create_agent()` ist ein hervorragender Einstieg – aber es stößt bei komplexen
Szenarien an seine Grenzen.

In der Praxis tritt das Schleifen-Problem häufiger auf als erwartet: Ein Agent, der nach Tool-Ergebnissen weiterfragt, statt zu schließen, verbraucht Token und Zeit — oft wegen eines unklaren Abbruchkriteriums im Prompt.

<p><font color='black' size="5">
Die wichtigsten Grenzen eines einfachen Agenten
</font></p>

Ein `create_agent()`-Agent eignet sich für einzelne Aufgaben mit Tools. Für einen mehrstufigen, kontrollierbaren Ablauf braucht der Meeting-Briefing-Agent zusätzlich expliziten State, Routing, Wiederholung oder Human-in-the-Loop.


**Kurzregel-Übersicht**

Für einfache Agenten ohne persistenten Zustand reicht `create_agent()` vollständig aus. LangGraph wird nötig, sobald ein Meeting-Briefing-Agent Zustand, Routing, Wiederholung, Human-in-the-Loop oder explizite Kontrollpunkte braucht. `StateGraph` modelliert diese Abläufe über State, Nodes und Edges; bedingte Ablauflogik wird mit `add_conditional_edges()` abgebildet.

**Einordnung der bisherigen Muster:**

LangGraph verbindet Agent und Chain: Der Agent bringt die Entscheidung ein, die Chain liefert kontrollierte lineare Schritte, und der Graph legt fest, wie Entscheidungen, Zustand und Rücksprünge über mehrere Schritte organisiert werden.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsregel</font> </br></p>


diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    FRAGE{"Welche Komplexität
hat der Meeting-Briefing-Agent?"}
    FRAGE -->|"Einfach
kein Zustand"| CA["create_agent()
ausreichend"]
    FRAGE -->|"Zustand /
Session"| LG1["LangGraph
+ StateGraph"]
    FRAGE -->|"Verzweigungen /
Routing"| LG2["LangGraph
+ conditional_edges"]
    FRAGE -->|"user-in-
the-Loop"| LG3["LangGraph
+ interrupt"]
    FRAGE -->|"Mehrere
Agenten"| LG4["LangGraph
+ Subgraphen"]

    style FRAGE fill:#FF9800,color:#fff
    style CA    fill:#4CAF50,color:#fff
    style LG1   fill:#2196F3,color:#fff
    style LG2   fill:#2196F3,color:#fff
    style LG3   fill:#2196F3,color:#fff
    style LG4   fill:#2196F3,color:#fff
'''
mermaid(diagram, width=1000)



<p><font color='black' size="5">
Demonstration: Das Gedächtnis-Problem
</font></p>


Ein `create_agent()`-Agent modelliert ohne zusätzliche Übergabe oder Checkpointer keinen persistenten Workflow-State.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from genai_lib.model_config import WORKER
# Demo: create_agent() kann Tools nutzen, verwaltet aber keinen expliziten Workflow-State.
_briefing_question_log = {"fragen": []}

@tool
def note_briefing_question(frage: str) -> str:
    """Notiert eine Briefing-Frage in einem Demo-Protokoll und gibt die Anzahl notierter Fragen zurück."""
    _briefing_question_log["fragen"].append(frage)
    return f"Notierte Fragen: {len(_briefing_question_log['fragen'])}"

llm = init_chat_model(WORKER)
system_prompt = (
    "Rolle: Meeting-Briefing-Agent für den State-Test. "
    "Tool-Aufrufe korrekt ausführen und knapp auf Deutsch antworten."
)
agent = create_agent(model=llm, tools=[note_briefing_question], system_prompt=system_prompt)

_briefing_question_log["fragen"] = []
r1 = agent.invoke({"messages": [HumanMessage(content="Notiere diese Briefing-Frage: Warum verbessert RAG die Zuverlässigkeit?")]}, config={**run_cfg, "run_name": "m07_kap1_r1"})
print("Nach Aufruf 1:", r1["messages"][-1].content[:120])

r2 = agent.invoke({"messages": [HumanMessage(content="Welche Briefing-Frage wurde gerade notiert?")]}, config={**run_cfg, "run_name": "m07_kap1_r2"})
print("Nach Aufruf 2:", r2["messages"][-1].content[:120])
print()
print("Demo-Protokoll:", _briefing_question_log["fragen"])
print("Problem: create_agent() organisiert Tool-Nutzung, aber keinen expliziten, prüfbaren Workflow-State zwischen Schritten.")


**Was passiert hier?**

1. `create_agent(...)` — erstellt einen Agenten mit Modell, Tools und System-Prompt

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>
diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TB


    subgraph langgraph["LangGraph - Expliziter State"]
        direction TB
        V1["Frage"] --> N1["analyse_node"]
        N1 --> S[("State
frage, analyse, risiko")]

        S --> N2["routing_node"]
        N2 --> S
        S --> N3["antwort_node"]
    end


    subgraph create_agent["create_agent() - Tool-Agent ohne expliziten Workflow-State"]
        direction LR
        U1["Aufruf 1"] --> A1["Agent"]
        A1 <--> T1["Tool"]
        A1 --> R1["Antwort"]
        R1 --> X1(("?"))

        U2["Aufruf 2"] --> A2["Agent"]
        A2 --> R2["Antwort
Kontext nur über Nachrichten"]

        X1 -. "Workflow-State nicht modelliert" .-> A2
    end

    style X1 fill:#FF6B6B,stroke:#333,color:#fff
    style S  fill:#90EE90,stroke:#333,color:#000
    style create_agent fill:#fff5f5,stroke:#ff6b6b,stroke-dasharray: 5 5
    style langgraph fill:#f5fff5,stroke:#90ee90
'''
mermaid(diagram, width=1200)


# 2 | State Machine Konzept
---

LangGraph basiert auf dem Konzept der **State Machine** (Zustandsautomat).  
Ein Zustandsautomat beschreibt ein System durch:

| Begriff | Erklärung | LangGraph-Entsprechung |
|---------|-----------|----------------------|
| **State** | Der aktuelle Zustand des Briefing-Workflows | `TypedDict` mit allen Daten |
| **Node** | Eine Aktion, die den Zustand verändert | Python-Funktion |
| **Edge** | Übergang von einem Node zum nächsten | `add_edge()` |
| **Conditional Edge** | Bedingter Übergang | `add_conditional_edges()` |
| **START** | Einstiegspunkt des Graphen | `START` Konstante |
| **END** | Ausstiegspunkt des Graphen | `END` Konstante |



**Der State als einzige Wahrheitsquelle**

Alle Nodes lesen vom State und schreiben in den State.  
Nodes kommunizieren **nicht direkt** miteinander – nur über den gemeinsamen State.

```
State = {
    "messages": [...],    # Gesprächsverlauf
    "schritt":  2,        # Aktueller Schritt
    "ergebnis": "...",    # Zwischen-/Endergebnis
}
```

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ─── State-Definition ────────────────────────────────────────────────────
class BriefingState(TypedDict):
    """State des Agenten – wird von allen Nodes geteilt und verändert."""
    messages: Annotated[list, add_messages]  # add_messages = Nachrichten akkumulieren
    step:  int                            # Zählt Ausführungsschritte
    done:   bool                           # Signal zum Beenden

# add_messages ist ein Reducer:
# Statt list zu überschreiben, werden neue Nachrichten ANGEHÄNGT
print("State-Typen:")
print(f"  messages → list (akkumuliert via add_messages)")
print(f"  step  → int  (überschrieben)")
print(f"  done   → bool (überschrieben)")

# Initialer State
start_state: BriefingState = {
    "messages": [],
    "step":  0,
    "done":   False,
}
print(f"\nStart-State: {start_state}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    ST([START]) --> N1

    N1["Node: llm_aufrufen()\n• liest: messages\n• schreibt: messages, step"] --> COND

    COND{"Bedingung:\nfertig == True?"}
    COND -->|"Nein"| N2
    COND -->|"Ja"| FIN

    N2["Node: tool_ausführen()\n• liest: messages\n• schreibt: messages"] --> N1

    FIN([FINISH])

    STATE[/"State (geteilt):\nmessages: [...]\nschritt: 2\nfertig: False"/]

    N1 <-.->|lesen/schreiben| STATE
    N2 <-.->|lesen/schreiben| STATE

    style ST fill:#90EE90
    style FIN fill:#FFB6C1
    style COND fill:#FFD700
    style STATE fill:#E0E0FF,stroke:#9999CC
'''

mermaid(diagram, width=450)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

llm = init_chat_model(BASELINE)

def llm_node(state: BriefingState) -> dict:
    """Ruft das LLM mit dem aktuellen Nachrichtenstand auf."""
    response = llm.invoke(state["messages"], config={**run_cfg, "run_name": "m07_kap2_llm_node"})
    return {
        "messages": [response],
        "step":  state["step"] + 1,
        "done":   True,
    }

agent_builder = StateGraph(BriefingState)
agent_builder.add_node("llm", llm_node)
agent_builder.add_edge(START, "llm")
agent_builder.add_edge("llm", END)
agent = agent_builder.compile()
print("Graph aufgebaut!")
print("Nodes:", list(agent_builder.nodes.keys()))


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


# 3 | Visualisierung: Graph-Diagramm
---

LangGraph kann den Graphen automatisch als Bild rendern – ideal zum Debuggen und zur Dokumentation.

```python
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))
```

> **Best Practice:** Immer nach `compile()` ausführen – zeigt den tatsächlichen,
> kompilierten Graphen (nicht nur den Builder).

> **💡 Tipp: Mermaid Live Editor**
> Eigene Graphen und Diagramme direkt im Browser entwerfen, ohne Code auszuführen:
> [https://mermaid.live](https://mermaid.live)
> Ideal zum Skizzieren der Graph-Struktur vor der Implementierung.

In [ ]:
# Graph nach compile() visualisieren (LangGraph Best Practice)
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
initial_state: BriefingState = {
    "messages": [HumanMessage(content="Was ist LangGraph für einen Meeting-Briefing-Agent in einem Satz?")],
    "step":  0,
    "done":   False,
}

ergebnis = agent.invoke(initial_state, config={**run_cfg, "run_name": "m07_kap3_agent_ergebnis"})

mprint(f"""
## Ergebnis des ersten Graphen

**Eingabe:** {initial_state["messages"][0].content}

**Antwort:** {ergebnis["messages"][-1].content}

**Schritte:** {ergebnis["step"]}

**Fertig:** {ergebnis["done"]}
""")


# 4 | LangGraph vs. LangChain
---

Beide Frameworks ergänzen sich:

| Situation | Passendes Muster |
|---|---|
| Einzelne Aufgabe, feste Chain oder einfacher Tool-Agent | LCEL oder `create_agent()` |
| Mehrere Schritte, gemeinsamer State, Routing, Schleifen oder Freigaben | `StateGraph` |

**Merksatz:** `create_agent()` kapselt den Agenten-Loop. `StateGraph` macht den Workflow und seine Kontrollpunkte explizit.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    Q["Neue Briefing-Agent-\nAufgabe"]
    Q --> A{"Zustand zwischen
Schritten nötig?"}
    A -->|"Nein"| B{"Routing
vorhersehbar?"}
    A -->|"Ja"| LG
    B -->|"Ja"| C{"user Review
oder Multi-Agent?"}
    B -->|"Muss explizit sein"| LG
    C -->|"Nein"| CA["LangChain \ncreate_agent()"]
    C -->|"Ja"| LG["LangGraph"]
    style CA fill:#90EE90
    style LG fill:#87CEEB
    style Q fill:#FFD700
'''
mermaid(diagram, width=450)


> **📚 Weiterführend: Modell-Auswahl pro Knoten**
>
> Ab Tag 5 (*Multi-Agent Patterns*) wird relevant, **welches Modell in welchem Knoten** eingesetzt wird:
> Supervisor- und Routing-Knoten brauchen stärkere Reasoning-Modelle als Worker-Knoten.
>
> → [Modell-Auswahl Guide](https://ralf-42.github.io/Agenten/03-modelle-provider-anpassung/modellauswahl.html)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Den **State** des einfachen Graphen aus Kapitel 3 beobachten.

- Den Graphen `agent` mit einer eigenen Meeting- oder Briefing-Frage aufrufen.
- Den Startzustand mit dem Ergebnis-State vergleichen: `schritt`, `fertig` und die Anzahl der `messages`.
- Festhalten, dass `llm_node` die Antwort ergänzt, `schritt` erhöht und `fertig` auf `True` setzt.


In [ ]:
# Grundlagen: State vorher/nachher vergleichen
# Startpunkt: agent und BriefingState aus Kapitel 2

# 1. Eigenen Startzustand (initial_state) mit einer Briefing-Frage anlegen
# 2. agent.invoke(initial_state) aufrufen (grundlagen_ergebnis)
# 3. In state_vergleich für 'step', 'done' und 'anzahl_messages' jeweils den Wert vorher und nachher festhalten


**Aufbau**

Die **Entscheidungsregel** aus Kapitel 1 auf Briefing-Szenarien anwenden.

- Szenario 1: Der Agent beantwortet eine einzelne Briefing-Frage und ruft bei Bedarf ein Tool auf.
- Szenario 2: Der Workflow soll den Zwischenstand einer Briefing-Frage ausdrücklich an einen weiteren Verarbeitungsschritt übergeben.
- Szenario 3: Vor dem Versand eines Briefings muss ein Mensch bestätigen.
- Für jedes Szenario `create_agent()` oder `LangGraph` wählen und mit einem der vier Hauptprobleme aus Kapitel 1 begründen.


In [ ]:
# Aufbau: Entscheidungsregel anwenden
# Startpunkt: Tabelle 'Die 4 Hauptprobleme' und Entscheidungsregel aus Kapitel 1

szenario_einordnung = {
    1: {"werkzeug": "", "problem_nr": None, "begruendung": ""},
    2: {"werkzeug": "", "problem_nr": None, "begruendung": ""},
    3: {"werkzeug": "", "problem_nr": None, "begruendung": ""},
}

# Pro Szenario: werkzeug = "create_agent()" oder "LangGraph", problem_nr = 1 bis 4 (falls zutreffend, sonst None),
# begruendung = ein Satz mit mindestens 20 Zeichen


**Vertiefung**

Den **State zwischen zwei Graph-Aufrufen ausdrücklich weiterreichen**. Das Beispiel zeigt dabei keine dauerhafte Speicherung und keinen Qualitäts- oder Routing-Loop.

- Den vorhandenen Graphen mit `llm_node` und den State-Feldern `messages`, `schritt` und `fertig` wiederverwenden.
- Aufruf 1 mit einem Meeting- oder Briefing-Thema ausführen und das Ergebnis als `ergebnis_1` speichern.
- Für Aufruf 2 einen neuen State bilden: die bisherigen `messages` übernehmen, eine zweite HumanMessage anhängen und `schritt` aus `ergebnis_1` weiterreichen.
- Aufruf 2 mit diesem neuen State ausführen und die beiden Ergebnisse vergleichen.
- Kurz festhalten: LangGraph übernimmt den State innerhalb eines Laufs; bei getrennten Aufrufen muss der aktuelle State von der Anwendung weitergereicht werden.


In [ ]:
# Vertiefung: State zwischen zwei Graph-Aufrufen weiterreichen
# Startpunkt: BriefingState und llm_node aus Kapitel 2 wiederverwenden

# 1. Einen Graphen mit llm_node bauen und kompilieren (graph_gedaechtnis)
# 2. Aufruf 1 mit einem Meeting- oder Briefing-Thema ausführen (ergebnis_1)
# 3. Für Aufruf 2 die bisherigen messages übernehmen und eine neue HumanMessage anhängen
# 4. step aus ergebnis_1 weiterreichen und den neuen State ausführen (ergebnis_2)
# 5. In _state_beobachtung (>= 20 Zeichen) festhalten, dass die Anwendung den State zwischen getrennten Aufrufen weitergibt


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangGraph - Überblick](https://ralf-42.github.io/Agenten/05-frameworks/langgraph.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
- [LangGraph Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langgraph-best-practices.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
